mount your data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


display  data

In [ ]:
import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/gbv_data.xlsx - Sheet1.csv")

In [ ]:
data

,1,The support group has been incredibly helpful and supportive
0,2.0,I finally feel safe and understood after joini...
1,3.0,The legal aid I received was prompt and effect...
2,4.0,My friends and family have been my biggest sup...
3,5.0,I am grateful for the help I received in findi...
4,6.0,The community center provided excellent resour...
...,...,...
112,NaN,I feel betrayed by those who should have prote...
113,NaN,I can't seem to escape the shadows of my past.
114,NaN,I am haunted by flashbacks and nightmares.
115,NaN,The constant vigilance is wearing me down.


Data Preprocessing

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Preprocessing function
def preprocess(text):
    tokens = word_tokenize(text)
    tokens = [word.lower() for word in tokens if word.isalpha() and word.lower() not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(tokens)

    # Apply preprocessing to the text column
data['processed_text'] = data['The support group has been incredibly helpful and supportive'].apply(preprocess)
print(data.head())


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


     1 The support group has been incredibly helpful and supportive  \
0  2.0  I finally feel safe and understood after joini...             
1  3.0  The legal aid I received was prompt and effect...             
2  4.0  My friends and family have been my biggest sup...             
3  5.0  I am grateful for the help I received in findi...             
4  6.0  The community center provided excellent resour...             

                                      processed_text  
0  finally feel safe understood joining counselin...  
1                legal aid received prompt effective  
2               friend family biggest support system  
3         grateful help received finding good lawyer  
4  community center provided excellent resource r...  


Emotion and Sentiment Analysis

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

# Initialize VADER sentiment analyzer
sid = SentimentIntensityAnalyzer()

# Function to get sentiment scores
def get_sentiment(text):
    return sid.polarity_scores(text)

# Apply sentiment analysis
data['sentiment'] = data['The support group has been incredibly helpful and supportive'].apply(get_sentiment)
print(data.head())




     1 The support group has been incredibly helpful and supportive  \
0  2.0  I finally feel safe and understood after joini...             
1  3.0  The legal aid I received was prompt and effect...             
2  4.0  My friends and family have been my biggest sup...             
3  5.0  I am grateful for the help I received in findi...             
4  6.0  The community center provided excellent resour...             

                                      processed_text  \
0  finally feel safe understood joining counselin...   
1                legal aid received prompt effective   
2               friend family biggest support system   
3         grateful help received finding good lawyer   
4  community center provided excellent resource r...   

                                           sentiment  
0  {'neg': 0.0, 'neu': 0.756, 'pos': 0.244, 'comp...  
1  {'neg': 0.0, 'neu': 0.566, 'pos': 0.434, 'comp...  
2  {'neg': 0.0, 'neu': 0.58, 'pos': 0.42, 'compou...  
3  {'neg': 0.0, 

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

# Initialize emotion detection pipeline
emotion_classifier = pipeline('sentiment-analysis', model='j-hartmann/emotion-english-distilroberta-base')

# Ensure the dataset has a 'text' column
if 'The support group has been incredibly helpful and supportive' not in data.columns:
    raise ValueError("Dataset must contain a 'The support group has been incredibly helpful and supportive' column")

# Function to get emotions
def get_emotions(text):
    return emotion_classifier(text)[0]['label']

# Apply emotion detection in batches
batch_size = 32  # Adjust batch size based on your memory and processing power
emotions = []

for i in tqdm(range(0, len(data), batch_size)):
    batch_texts = data['The support group has been incredibly helpful and supportive'][i:i+batch_size].tolist()
    batch_emotions = [get_emotions(text) for text in batch_texts]
    emotions.extend(batch_emotions)

# Add the emotions to the DataFrame
data['emotions'] = emotions

# Print the result
print(data)


100%|██████████| 4/4 [00:09<00:00,  2.28s/it]

       1 The support group has been incredibly helpful and supportive  \
0    2.0  I finally feel safe and understood after joini...             
1    3.0  The legal aid I received was prompt and effect...             
2    4.0  My friends and family have been my biggest sup...             
3    5.0  I am grateful for the help I received in findi...             
4    6.0  The community center provided excellent resour...             
..   ...                                                ...             
112  NaN  I feel betrayed by those who should have prote...             
113  NaN     I can't seem to escape the shadows of my past.             
114  NaN         I am haunted by flashbacks and nightmares.             
115  NaN         The constant vigilance is wearing me down.             
116  NaN           I feel like I will never be whole again.             

                                        processed_text  \
0    finally feel safe understood joining counselin...   
1      

Contextual Analysis

In [ ]:
from gensim import corpora
from gensim.models.ldamodel import LdaModel

# Create a dictionary representation of the documents
dictionary = corpora.Dictionary(data['processed_text'].apply(str.split))

# Create a corpus (a list of bags of words)
corpus = [dictionary.doc2bow(text.split()) for text in data['processed_text']]

# Train the LDA model
lda_model = LdaModel(corpus, num_topics=3, id2word=dictionary, passes=15)

# Print the topics
topics = lda_model.print_topics(num_words=4)
for topic in topics:
    print(topic)

(0, '0.031*"feel" + 0.029*"incident" + 0.026*"support" + 0.022*"legal"')
(1, '0.062*"feel" + 0.021*"like" + 0.020*"life" + 0.013*"new"')
(2, '0.031*"support" + 0.018*"center" + 0.014*"fear" + 0.014*"provided"')


Response Generation

In [ ]:
import pandas as pd
from transformers import pipeline
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# Initialize emotion detection pipeline
emotion_classifier = pipeline('sentiment-analysis', model='j-hartmann/emotion-english-distilroberta-base')


# Ensure the dataset has a 'text' column
if 'The support group has been incredibly helpful and supportive' not in data.columns:
    raise ValueError("Dataset must contain a 'The support group has been incredibly helpful and supportive' column")

# Function to get emotions
def get_emotions(text):
    return emotion_classifier(text)[0]['label']

# Process data in parallel
def process_batch(texts):
    return [get_emotions(text) for text in texts]

batch_size = 32  # Adjust batch size based on your memory and processing power
num_workers = 4  # Adjust the number of workers based on your CPU cores

emotions = []

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = []
    for i in range(0, len(data), batch_size):
        batch_texts = data['The support group has been incredibly helpful and supportive'][i:i+batch_size].tolist()
        futures.append(executor.submit(process_batch, batch_texts))

    for future in tqdm(futures):
        emotions.extend(future.result())

# Add the emotions to the DataFrame
data['emotions'] = emotions

# Print the result
print(data.head())


100%|██████████| 4/4 [00:07<00:00,  1.76s/it]

     1 The support group has been incredibly helpful and supportive  \
0  2.0  I finally feel safe and understood after joini...             
1  3.0  The legal aid I received was prompt and effect...             
2  4.0  My friends and family have been my biggest sup...             
3  5.0  I am grateful for the help I received in findi...             
4  6.0  The community center provided excellent resour...             

                                      processed_text  \
0  finally feel safe understood joining counselin...   
1                legal aid received prompt effective   
2               friend family biggest support system   
3         grateful help received finding good lawyer   
4  community center provided excellent resource r...   

                                           sentiment emotions  
0  {'neg': 0.0, 'neu': 0.756, 'pos': 0.244, 'comp...      joy  
1  {'neg': 0.0, 'neu': 0.566, 'pos': 0.434, 'comp...      joy  
2  {'neg': 0.0, 'neu': 0.58, 'pos': 0.42, 'c

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import pickle

# Initialize VADER sentiment analyzer
sid = SentimentIntensityAnalyzer()

# Save the model using pickle
with open('vader_sentiment_analyzer.pkl', 'wb') as f:
    pickle.dump(sid, f)


In [ ]:
from transformers import pipeline

# Initialize emotion detection pipeline
emotion_classifier = pipeline('sentiment-analysis', model='j-hartmann/emotion-english-distilroberta-base')

# Save the model using the save_pretrained method
emotion_classifier.model.save_pretrained('emotion_detection_model')


In [ ]:
from gensim import corpora
from gensim.models.ldamodel import LdaModel

# Save the LDA model
lda_model.save('lda_model')
